# Unit 04 - Introduction to Federated Learning

**Authors**:

- Patricia Guadalupe Alvarenga Mairena
- Laura González Lemos
- Iria Janeiro Pazos
- Ayesha Munir 
- Andrea Real Blanco


## Learning objectives

By the end of this unit, you should be able to:

- Explain the difference between centralized learning and federated learning
- Identify the roles of the **server** and the **clients** in Federated Learning
- Describe the Federated Learning training loop (communication rounds)
- Explain why Federated Learning is **not always preferable** to centralized learning
- Understand the role of Flower in a federated learning workflow

## Assumptions and scope

We assume prior knowledge of:
- Supervised learning and model evaluation
- Basic Python-based machine learning workflows

We do **not** assume:
- Background in distributed systems
- Prior experience with Federated Learning

Out of scope for this unit:
- Differential Privacy
- Secure aggregation and cryptographic protocols
- Production-level deployment considerations

# Introduction to Federated Learning

Federated Learning (FL) is a distributed machine learning paradigm in which multiple participants collaboratively train a shared global model without exchanging their raw data. Unlike traditional centralized learning approaches, the data remains local to each participant, and only model-related information is communicated during the training process.

Federated learning was first proposed by Google in 2016 and formally introduced in the work of McMahan et al. (2017). Since then, it has attracted significant attention and has been successfully applied in a wide range of domains, including healthcare (Hard et al., 2018), finance(Yoon et al., 2018), and natural language processing (Li et al., 2020), among others.

This paradigm has gained relevance due to the increasing importance of data privacy, regulatory constraints, and the natural distribution of data across organizations and devices. In many real-world scenarios—such as collaborative medical studies or large-scale mobile applications—centralizing data is either undesirable or impractical.

In a typical federated learning setup, a central server orchestrates the learning process by coordinating multiple clients. Each client performs local training using its own data and periodically sends model updates to the server, which aggregates them to produce an improved global model. This global model is then redistributed to the clients, and the process is repeated over several communication rounds.

Despite its advantages, federated learning introduces challenges that do not appear in centralized settings. Data across clients is often heterogeneous and non-identically distributed, communication between clients and server can be costly, and not all clients may be available at every training round.

In this notebook, we will introduce the fundamental concepts of federated learning and simulate a simple federated scenario. The goal is to build an intuitive understanding of how federated learning differs from centralized training, laying the foundation for implementing and analyzing a complete federated learning algorithm in the next session.



# Centralized Learning vs. Federated Learning

In traditional centralized machine learning, all training data is collected and stored in a single location, where the model is trained using the complete dataset. This approach has been the dominant paradigm for many years and underlies most standard machine learning pipelines, as it simplifies both model training and evaluation.

Federated learning departs from this paradigm by keeping the data decentralized. Instead of transferring data to a central server, the learning process is distributed across multiple clients, each of which trains a local model using its own data. The central server is responsible for coordinating the process and aggregating the information received from the clients in order to update a shared global model.

From a methodological perspective, centralized learning benefits from direct access to the full data distribution, which often leads to stable optimization and straightforward convergence behavior. In contrast, federated learning must cope with data that is fragmented across clients and often follows heterogeneous, non-identically distributed patterns. As a result, optimization becomes more challenging and model updates may reflect local biases present in individual clients.

Beyond methodological considerations, the two paradigms differ significantly in terms of data governance and system design. Centralized learning requires data transfer and storage in a single repository, which may raise privacy, legal, or ethical concerns. Federated learning, on the other hand, reduces the need for raw data sharing and is therefore better suited to scenarios where data cannot be centralized due to regulatory constraints or institutional boundaries.

Understanding these differences is essential for appreciating both the potential and the limitations of federated learning. In the following sections, we will simulate a federated setting to highlight how decentralization and data heterogeneity affect the learning process in


# Key Components of a Federated Learning System

A federated learning system is composed of several fundamental components that interact throughout the training process. Understanding these components is essential before examining how the federated learning algorithm operates over time.

- **Client**  
  A client refers to a device or edge node that holds a local dataset and actively participates in the training of the federated model. Each client performs local computation using its private data and communicates model updates to the server.

- **Server**  
  The server is the central coordinating entity responsible for managing the federated learning process. It distributes the global model to selected clients, receives their model updates, and aggregates them to produce a new version of the global model.

- **Federated dataset**  
  The federated dataset consists of multiple decentralized local datasets, each owned by a different client. These datasets are not directly accessible by the server and are used collaboratively to train the federated model.

- **Federated model**  
  The federated model is the machine learning model that is collaboratively trained using the federated dataset. Although training is performed in a decentralized manner, the goal is to obtain a global model that generalizes well to new data while preserving the privacy of each client’s local data.

- **Federated optimization**  
  Federated optimization refers to the process of training the federated model using decentralized data and model updates from the clients. This optimization setting differs from classical centralized optimization due to data heterogeneity, partial client participation, and communication constraints.

- **Aggregation**  
  Aggregation is the process by which the server combines the model updates received from multiple clients into a new global model. This step is crucial for incorporating information learned across clients and is commonly implemented using weighted averaging or related techniques.

- **Rounds**  
  Rounds refer to the iterative cycles of communication in federated learning. In each round, the global model is distributed to clients, locally updated, aggregated, and redistributed. Training proceeds over multiple rounds until a stopping criterion is reached.

With these components in mind, we can now describe the typical phases that govern the execution of a federated learning algorithm.



# Phases of a Federated Learning Algorithm

Once the main components of a federated learning system have been defined, the training process can be described as a sequence of iterative phases that are repeated over multiple communication rounds. These phases determine how the federated model is collaboratively trained while keeping the data decentralized.

1. **Initialization of the global model**  
   The federated learning process starts with the initialization of a global model on the server. This model serves as the common starting point for all clients and may be initialized randomly or using pre-trained weights.

2. **Distribution of the global model to clients**  
   At the beginning of each round, the server selects a subset of available clients and sends them the current version of the global model. Client selection strategies may vary depending on system constraints, availability, or efficiency considerations.

3. **Local training on client data**  
   Each selected client performs local optimization of the received model using its own federated dataset. Training is typically carried out for a limited number of local epochs to balance learning progress and communication cost.

4. **Communication of local updates to the server**  
   After completing local training, clients transmit their model updates back to the server. These updates may consist of the full model parameters or parameter differences relative to the received global model.

5. **Aggregation of client updates**  
   The server aggregates the updates received from the clients to produce a new global model. This aggregation step integrates information learned across the decentralized datasets and is commonly implemented using weighted averaging based on the size of each client’s dataset.

6. **Iteration over communication rounds**  
   The updated global model is redistributed to the clients, and the process is repeated over multiple rounds. Training continues until a stopping criterion is met, such as convergence of the global model or reaching a predefined number of rounds.

This iterative procedure defines the core workflow of most federated learning algorithms and will serve as the conceptual basis for the simulations and implementations explored in this notebook.



# Introduction to Flower (Flwr)

In order to experiment with federated learning algorithms in practice, it is necessary to rely on software frameworks that abstract away much of the complexity associated with distributed systems, client coordination, and communication. One such framework is Flower (Flwr), a flexible and lightweight open-source framework specifically designed for federated learning.

It is important to emphasize that Flwr is not a machine learning algorithm. Instead, it is a software framework that provides the infrastructure required to implement, simulate, and deploy federated learning algorithms. The learning algorithm itself—such as the model architecture, loss function, and optimization strategy—remains under the control of the user.

Flwr follows a modular and framework-agnostic design, allowing developers and researchers to integrate federated learning workflows with popular machine learning libraries such as PyTorch, TensorFlow, or NumPy. Rather than enforcing a specific algorithmic formulation, Flwr enables users to define custom client behavior, server-side logic, and aggregation strategies.

From a conceptual standpoint, Flwr closely reflects the components and phases discussed in the previous sections. It explicitly models the interaction between a central server and multiple clients, supports iterative communication rounds, and facilitates the exchange and aggregation of model updates. This clear correspondence between theory and implementation makes Flwr particularly well suited for educational and experimental purposes.

In this notebook, Flwr will be used to simulate a federated learning environment on a single machine. This approach allows us to focus on the fundamental principles of federated learning without requiring a fully distributed infrastructure. In the following sections, we will introduce the basic elements of Flwr and use them to construct a simple federated learning scenario.

To use Flower for federated learning experiments, the library must first be installed and imported in the Python environment:

In [1]:
# Attempt to import Flower (Flwr). If it is not installed, install it in simulation mode.
try:
    import flwr as fl
except ImportError:
    !pip install flwr[simulation]
    !pip install ray[default]  # Required dependency for Flower in simulation mode
    import flwr as fl  # Import again after installation

# Attempt to import TensorFlow. Install it if not available.
try:
    import tensorflow as tf
except ImportError:
    !pip install tensorflow
    import tensorflow as tf  # Import again after installation


2026-03-20 16:14:15.368865: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-20 16:14:15.451565: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 16:14:17.416561: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


When setting up a federated learning simulation environment with Flower, it is important to install the framework using the `simulation` extra, that is, `flwr[simulation]`. This option ensures that all additional dependencies required to simulate a federated learning environment on a single machine are correctly installed.

In particular, Flower relies on the `flwr_simulation` module to orchestrate multiple simulated clients within a single Python process. Additionally, starting from version 1.15, Flower uses `ray` as a backend for its simulation mode. For this reason, `ray` must also be available in the environment when running federated learning simulations.

It is worth noting that this setup is specific to simulation-based experiments. In a fully distributed deployment, where the server and clients run on separate machines, Flower should be installed using `pip install flwr` on both the server and all client devices to ensure compatibility and consistency across the system.

For the most up-to-date and comprehensive installation instructions, refer to the official Flower documentation:[Link to official Flower documentation](https://flower.ai/)

After installing the `flwr` package, it can be imported into Python as shown below.

In [2]:
import flwr as fl
import tensorflow as tf

Flower provides a set of classes and abstractions that can be used to define a federated learning environment, coordinate training across clients, and evaluate a global model. Rather than prescribing a specific machine learning algorithm, Flower allows users to integrate their own models and training logic using standard machine learning libraries.

An important practical consideration in federated learning is that the model must be serializable, as model parameters need to be transmitted between the server and the clients during training. As a consequence, not all model types are equally suitable for federated learning, particularly in simulation or distributed settings.

In this notebook, we will use an Artificial Neural Network (ANN) implemented with TensorFlow and Keras. Keras models are lightweight, widely used, and well-suited for serialization, making them a convenient choice for illustrating federated learning concepts.

For more details on how Flower integrates with TensorFlow and Keras, refer to the official Flower documentation:
https://flower.dev/docs/quickstart-tensorflow.html

We now define a simple neural network model that each federated client will use.

In [3]:
# Define a simple neural network model using TensorFlow and Keras
def generate_ann():
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(32, 32, 3)),# Input: 32x32 RGB images
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(64, activation="relu"),     # Hidden layer
            tf.keras.layers.Dense(64, activation="relu"),     # Hidden layer
            tf.keras.layers.Dense(10, activation="softmax"),  # Output: 10 classes
        ]
    )

    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )

    return model

> At this point, the model definition is identical to what we would use in centralized learning. What do you expect to change when moving to a federated setting?

In a federated learning, a communication protocol must be established between the server and the client to share models and manage data partitions. Moreover, aggregation strategies to merge updates into a global model are also needed.

As we will train a deep learning model with TensorFlow, it is convenient to work with `tf.data.Dataset` objects, since they enable efficient input pipelines and can take advantage of hardware acceleration (e.g., GPUs) when available.

However, in Flower simulation mode it is often easier (and more robust) to keep the *data containers exchanged between components* as simple and serializable Python objects (e.g., NumPy arrays or tuples). For that reason, in this notebook we will first load and partition the data manually into per-client splits using NumPy arrays. Later, each client will convert its local split into `tf.data.Dataset` objects for training and evaluation.


In [4]:
import numpy as np
import tensorflow as tf

NUM_CLIENTS = 3
RANDOM_SEED = 42

def unison_shuffled_copies(a: np.ndarray, b: np.ndarray, seed: int = RANDOM_SEED):
    """Shuffle two arrays in unison, preserving alignment between inputs and labels."""
    assert len(a) == len(b)
    rng = np.random.default_rng(seed)
    p = rng.permutation(len(a))
    return a[p], b[p]

def split_index(a: np.ndarray, n: int):
    """Return a list of index arrays splitting 'a' into 'n' approximately equal parts."""
    return np.array_split(np.arange(len(a)), n)

def load_datasets(num_clients: int, train_size: int = 10_000, test_size: int = 1_000):
    """
    Load CIFAR-10, normalize, shuffle, and split it into per-client (train, val, test) tuples.

    Returns:
        train_splits: list of (x_train_i, y_train_i) for each client
        val_splits:   list of (x_val_i,   y_val_i)   for each client
        test_splits:  list of (x_test_i,  y_test_i)  for each client
    """
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    # Normalize to [0, 1]
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    # Keep the notebook lightweight (faster for classroom execution)
    x_train, y_train = x_train[:train_size], y_train[:train_size]
    x_test, y_test = x_test[:test_size], y_test[:test_size]

    # Flatten labels from shape (N, 1) to (N,) to simplify downstream handling
    y_train = y_train.squeeze()
    y_test = y_test.squeeze()

    # Shuffle (fixed seed for reproducibility in class)
    x_train, y_train = unison_shuffled_copies(x_train, y_train, seed=RANDOM_SEED)
    x_test, y_test = unison_shuffled_copies(x_test, y_test, seed=RANDOM_SEED + 1)

    # Split indices per client
    train_index = split_index(x_train, num_clients)
    test_index = split_index(x_test, num_clients)

    train_splits, val_splits, test_splits = [], [], []
    
    for cid in range(num_clients):
        client_train_idx = train_index[cid]
        client_test_idx = test_index[cid]

        # Per-client train split
        x_c, y_c = x_train[client_train_idx], y_train[client_train_idx]

        # 10% validation (kept simple and deterministic)
        val_size = max(1, len(x_c) // 10)
        x_val, y_val = x_c[:val_size], y_c[:val_size]
        x_tr, y_tr = x_c[val_size:], y_c[val_size:]

        train_splits.append((x_tr, y_tr))
        val_splits.append((x_val, y_val))
        test_splits.append((x_test[client_test_idx], y_test[client_test_idx]))

    return train_splits, val_splits, test_splits

trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

/home/arebla/OneDrive/MSc/ML2/venv/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Before proceeding to a federated learning setup, it is good practice to validate the data pipeline and model definition in a simpler setting. As a sanity check, we first train and evaluate the model using the local dataset of a single client. It is like a centralized train to check the correctness.

This step serves two main purposes. First, it verifies that the model can be successfully trained using the client’s data, ensuring that the model architecture, loss function, and optimizer are correctly defined. Second, it helps identify potential issues in the data loading and preprocessing steps early, before introducing the additional complexity of federated learning.


In [5]:
# Instantiate a fresh model
model = generate_ann()

# Select the data of a single client (client_id = 0)
x_train_c, y_train_c = trainloaders[0]
x_val_c, y_val_c = valloaders[0]

# Train the model locally (single-client, centralized setting)
model.fit(
    x_train_c,
    y_train_c,
    epochs=1,
    batch_size=32,
    verbose=1,
)

# Evaluate on the client's validation set
loss, accuracy = model.evaluate(x_val_c, y_val_c, verbose=0)

print(f"Validation loss: {loss:.4f}")
print(f"Validation accuracy: {accuracy * 100:.2f}%")


2026-03-20 16:14:23.312872: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1789 - loss: 2.2025
Validation loss: 2.0030
Validation accuracy: 27.03%


At this point, we move from local, single-client training to a federated learning setting. Conceptually, the local training procedure remains the same, but it is now executed independently by multiple clients and coordinated by a central server.

In this notebook, we use **Flower in simulation mode**, which allows us to simulate the interaction between a server and multiple clients on a single machine. This approach avoids the need to launch separate processes or machines, while preserving the logical structure of a federated learning system.

In Flower, the **server** is responsible for coordinating the training process. It selects clients for each communication round, distributes the current global model parameters, and aggregates the updates returned by the clients. Each **client** performs local training using its private dataset and reports the results back to the server.

The interaction between server and clients is defined through a client interface. When a client is selected during a training round, the server invokes specific methods on the client to carry out local training and evaluation. These method calls conceptually correspond to network communication in a fully distributed setup, but are simulated locally in this notebook.

For TensorFlow and Keras-based workloads, Flower provides a convenient base class called `NumPyClient`. This class simplifies the implementation of the client interface by allowing model parameters to be exchanged as NumPy arrays. A `NumPyClient` typically implements three methods:

**Note:** Flower supports both simulation-based experiments and fully distributed deployments. In this notebook, we exclusively work in **simulation mode**.

In [6]:
# Define a Flower client by extending NumPyClient
class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data  # tuple (x_train, y_train)
        self.val_data = val_data      # tuple (x_val, y_val)

    def get_parameters(self, config):
        # Return the current local model parameters
        return self.model.get_weights()

    def fit(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=1,
            batch_size=32,
            verbose=0,
        )

        # Return updated parameters and number of training examples
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        # Return loss, number of evaluation examples, and metrics
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

In Flower, clients can be created by extending either the `flwr.client.Client` or `flwr.client.NumPyClient` classes. In the previous example, we used `NumPyClient` because it is easier to implement and requires less code as a template. Along with the extended class, there are three main methods that need to be implemented:

* `get_parameters`: Returns the current local model parameters.
* `fit`: Receives model parameters from the server, trains the model parameters on the local data, and returns the (updated) model parameters to the server.
* `evaluate`: Receives model parameters from the server, evaluates the model parameters on the local data, and returns the evaluation result to the server.

As you can see, the `MyClient` class implemented in the previous example follows this same structure.


In simulation mode, Flower needs a way to instantiate a client given its identifier (`cid`). The function `client_fn` acts as a factory: for each simulated client, it selects the corresponding local data partition, creates a fresh model, and returns a `NumPyClient` instance.

A key detail is that each client should have its own model instance. Even though all clients share the same architecture, their parameters will evolve differently during local training before being aggregated on the server.

Sometimes, especially when we are simulating multiple clients on a single device, it can be useful to use a function to create the client when it is required. This is particularly important in stateless frameworks, such as PyTorch, which can benefit from a more efficient implementation that creates clients only when they are required for training or evaluation. For example, the following code loads different examples for each client before discarding them:


In [7]:
from flwr.common import Context

def client_fn(context: Context) -> fl.client.Client:
    """Create a Flower client for a given simulated node."""
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)


    # Each client gets its own model instance
    model = generate_ann()

    # Select this client's local data partition
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    # Return a Flower client
    return MyClient(model, train_data, val_data).to_client()

## Running Flower as an App (ClientApp + ServerApp)

Recent versions of Flower encourage organizing federated learning projects as a **Flower App**. In this setup, the client-side logic is wrapped in a `ClientApp` and the server-side orchestration is wrapped in a `ServerApp`. This structure is the recommended approach for both simulation and distributed deployments.

In simulation mode, we can execute the app directly from this notebook using `run_simulation(...)`, which runs a `ServerApp` together with multiple `ClientApp` instances (one per simulated node). To do so we would require the following code:


In [8]:
from flwr.clientapp import ClientApp

client_app = ClientApp(client_fn=client_fn)


## Server-side Strategy and Simulation
One point to highlight is that the framework is not only going to manage the `losses_distributed`, but none of the other metrics. Due to the diverse treatment of those measures, the framework cannot accurately handle the aggregation of these metrics. Users need to tell the framework how to handle and aggregate these custom metrics.

The strategy will then call these functions whenever it receives fit or evaluates metrics from clients. The two possible functions are `fit_metrics_aggregation_fn` and `evaluate_metrics_aggregation_fn`. For example, the following code creates the weighted average, and the previous example can be adapted as follows:


In [9]:
from typing import List, Tuple, Dict
from flwr.common import Metrics

def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    if not metrics:
        return {}

    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}



In Flower simulation mode, we do not explicitly start a server process. Instead, we configure the *server-side strategy* and let Flower orchestrate the full federated workflow (server coordination, client selection, parameter distribution, and aggregation) within a single Python process.

The most common baseline strategy is **Federated Averaging (FedAvg)**. At each round, the server selects a fraction of the available clients, sends them the current global model parameters, receives locally-updated parameters after training, and aggregates them (typically via a weighted average) to produce the next global model.

In the next cell, we configure a FedAvg strategy and start a federated learning simulation with `NUM_CLIENTS` clients for a small number of rounds.

In [10]:
from flwr.server import ServerApp, ServerAppComponents

num_rounds = 3

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    initial_weights = model.get_weights()
    del model  # free memory early

    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(initial_weights)

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=0.5,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    return ServerAppComponents(strategy=strategy, config=config)

# Create Server
server_app = ServerApp(server_fn=server_fn)


Once everything is implemented, we can run the simulation

In [11]:
from flwr.simulation import run_simulation

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated nodes/clients
)

2026-03-20 16:14:25,659	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
INFO :      Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
/home/arebla/OneDrive/MSc/ML2/venv/lib/python3.11/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(pid=154820) 2026-03-20 16:14:41.955660: I external/local_xla/xla/tsl/cuda/cudart

**Note**: *Be aware that the simulation is a really resource consuming task. You can get some errors and warnings linked to that. There are different [configurations](https://flower.ai/docs/framework/how-to-run-simulations.html) that you can apply to configure the available resources for simulating the process.*


## Inspecting the First Federated Learning Results

After running the federated learning simulation, we can analyse the results across the communication rounds. At this stage, the goal is not to achieve high accuracy, but to understand how the federated learning process behaves and how it differs from centralised training.

The `history` object returned by Flower should contain information about the evolution of the global model during training and evaluation. By inspecting these results, we can gain insight into how the model improves across rounds and how aggregation of client updates affects performance.

Depending on your version of flower, the results can be None in the return of the simulation, but for utility reasons, we can define the following function:

In [12]:
def print_history(h):
    if h is None:
        print("history is None (this Flower version may not return a History object from run_simulation).However, you can check the run_simulation logs")
        return

    # Distributed loss
    if getattr(h, "losses_distributed", None):
        print("Distributed loss:")
        for r, loss in h.losses_distributed:
            print(f"  Round {r}: loss = {loss:.4f}")

    # Distributed metrics
    if getattr(h, "metrics_distributed", None):
        print("\nDistributed metrics:")
        for metric_name, series in h.metrics_distributed.items():
            print(f"  {metric_name}:")
            for r, value in series:
                if isinstance(value, float):
                    print(f"    Round {r}: {value * 100:.2f}%")
                else:
                    print(f"    Round {r}: {value}")

print_history(history)

history is None (this Flower version may not return a History object from run_simulation).However, you can check the run_simulation logs


### Discussion

Several observations can be made from these initial results:

- The performance of the global model typically improves over communication rounds, although convergence may be slower than in centralized training.
- The accuracy obtained after a small number of rounds is often lower than what we observed during the single-client sanity check.
- Variability across clients and limited local training can lead to noisy or unstable updates, especially in early rounds.

These behaviors are expected and highlight some of the fundamental challenges of federated learning, such as data heterogeneity and limited communication. At this point, it is useful to reflect on how design choices—such as the number of rounds, the fraction of participating clients, or the amount of local training—may influence the final performance.


> **Questions for reflection:**
> - How do these results compare to the single-client experiment performed earlier?
> - Why might federated learning require more training rounds to reach similar performance?
> - What factors could explain differences in convergence speed or stability?


`Answer here`

- Comparison to single-client: Our federated learning results were generally weaker than the single-client experiment. In the single-client setting, training was faster and more stable because the model learned from one complete dataset directly. In federated learning, the data was split across clients, so the server only combined local updates, which made learning slower and less stable.
- Training rounds: Federated learning usually needs more rounds because the model does not train on all the data at once. Instead, each client trains locally for a short time, and then the server aggregates the updates. This makes the improvement more gradual, so more communication rounds are often needed to reach similar performance.
- Convergence speed and stability: The main factors are the distribution of data across clients (data heterogeneity), the number of local training steps, the number of participating clients, and the training hyperparameters. If clients have very different local data, their updates can be more inconsistent, which can slow down convergence and make training less stable.

**Exercise 1:** Increase the number of federated rounds and analyze how the accuracy evolves.

---

**Answer**: After modifying the `num_rounds` parameter, we can see that the performance of the global model improves over successive rounds (the loss keeps diminishing and the accuracy seems to converge to ~ 0.30), as it incorporates more information from the decentralized sources. Moreover, it can be seen that the accuracy fluctuates between rounds. This behavior is expected in federated learning because the global model is updated using aggregated client updates, which can introduce variability due to differences in local data distributions before we start to get a stable estimate of the gradient.

In [13]:
num_rounds = 10

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    initial_weights = model.get_weights()
    del model  # free memory early

    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(initial_weights)

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=0.5,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated nodes/clients
)

INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
(pid=158045) 2026-03-20 16:15:11.592765: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
(pid=158045) 2026-03-20 16:15:11.659145: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(pid=158045) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ClientAppActor pid=158040) 2026-03-20 16:15:16.456567: E external/local_xla/xla/stream_executor/cuda/cuda_platf

**Exercise 2:** Reduce `fraction_fit` to simulate partial client participation and observe the effect.

---

**Answer**: In this exercise, `fraction_fit` was reduced so that not all clients participated in every round. This simulates a more realistic federated learning setting, because in real systems some devices may be offline or unavailable.

Compared with full client participation, the learning process became slower and less stable because the server received updates from fewer clients at each round. This reduces the amount of information used to update the global model in every communication step, and makes the global model more sensitive to the specific data distributions of the selected clients.

The lower final accuracy, with respect to exercise 1, confirms that this limited communication hinders the model's ability to generalize. In this setting, the model still learns, but needs more time to reach the same performance.

In [14]:
num_rounds = 10
fraction_fit = 0.3

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    initial_weights = model.get_weights()
    del model  # free memory early

    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(initial_weights)

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=fraction_fit,
        fraction_evaluate=0.5,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated nodes/clients
)

INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
(pid=163399) 2026-03-20 16:15:56.049936: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
(pid=163399) 2026-03-20 16:15:56.111820: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(pid=163399) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ClientAppActor pid=163367) 2026-03-20 16:16:00.890440: E external/local_xla/xla/stream_executor/cuda/cuda_platf

**Exercise 3:** Compare federated learning results with centralized training using the union of all client datasets.

---

**Answer**: After training for 10 epochs using the centralized dataset, the model achieves a higher validation accuracy than the federated model after 10 communication rounds. This happens because in centralized learning the model is trained directly on the entire dataset at once, allowing the optimizer to compute gradients using the full dataset distribution.

In contrast, federated learning trains models locally on separate clients datasets and then aggregates their updates on the server. Because client data may follow different distributions and updates are averaged across rounds, the optimization process can be noisier and slower to converge. As a result, centralized training reaches higher accuracy more quickly than federated learning.

In [15]:
X_train_all = np.concatenate([trainloaders[i][0] for i in range(NUM_CLIENTS)])
y_train_all = np.concatenate([trainloaders[i][1] for i in range(NUM_CLIENTS)])
X_val_all = np.concatenate([valloaders[i][0]   for i in range(NUM_CLIENTS)])
y_val_all = np.concatenate([valloaders[i][1]   for i in range(NUM_CLIENTS)])
X_test_all = np.concatenate([testloaders[i][0]   for i in range(NUM_CLIENTS)])
y_test_all = np.concatenate([testloaders[i][1]   for i in range(NUM_CLIENTS)])

model = generate_ann()
print(f"{'='*50}\n TRAINING METRICS\n{'='*50}")
model.fit(X_train_all, y_train_all, validation_data=(X_val_all, y_val_all), epochs=10, batch_size=32)


# Evaluate on the whole test set
print(f"{'='*50}\n TEST METRICS\n{'='*50}")
loss, accuracy = model.evaluate(X_test_all, y_test_all, verbose=0)

print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

 TRAINING METRICS
Epoch 1/10


/home/arebla/OneDrive/MSc/ML2/venv/lib/python3.11/site-packages/keras/src/backend/tensorflow/core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)


282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.2469 - loss: 2.0639 - val_accuracy: 0.2743 - val_loss: 1.9514
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3215 - loss: 1.8714 - val_accuracy: 0.3403 - val_loss: 1.8110
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3513 - loss: 1.8034 - val_accuracy: 0.3343 - val_loss: 1.8373
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3646 - loss: 1.7599 - val_accuracy: 0.3463 - val_loss: 1.7888
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3808 - loss: 1.7211 - val_accuracy: 0.3463 - val_loss: 1.8039
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3896 - loss: 1.6889 - val_accuracy: 0.3584 - val_loss: 1.7922
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4066 - loss: 1.6602 - val_accuracy: 0.3964 - val_loss: 1.6794
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4132 - loss: 1.6275 - val_accuracy: 0.3844 - val_

**Exercise 4:** Limiting Local Computation per Round

In the current implementation, each client trains on its entire local dataset for one epoch at every communication round. In practice, federated learning systems often limit the amount of local computation per round to reduce computation and communication costs.

Modify the `fit` method of the client so that only a fixed number of training steps are performed per round using the `steps_per_epoch` argument. Then, rerun the simulation and analyze how this affects convergence and accuracy.

*Hint:* Try values such as `steps_per_epoch = 1`, `3`, or `5`, and compare the results with the baseline implementation.

---

**Answer**: 

`steps_per_epoch` | Final Accuracy | Final Loss |
| :--- | :--- | :--- |
| 1 | 35.34% | 1.780 |
| 3 | 35.53% | 1.768 |
| 5 | 38.59% | 1.728 |

As the number of training steps increases (from 1 to 5), the final accuracy improves significantly, reaching 38.59%. This indicates that performing more local optimization per round allows clients to transmit more informative updates to the server. 

Across all three settings, the accuracy shows a downward trend in later rounds (e.g., falling from 41.84% in round 1 to 38.58% in round 10 for the 5-step trial). This likely stems from the challenges of data heterogeneity and non-identically distributed patterns mentioned before. As local models optimize further on their specific partitions, they may cause the aggregated global model to lose generalization on the combined test set.

- steps_per_epoch=1
    ```
    Run finished 10 round(s) in 52.60s
    History (loss, distributed):
    	round 1: 1.647569179534912
    	round 2: 1.6581077178319295
    	round 3: 1.6593907276789348
    	round 4: 1.675180713335673
    	round 5: 1.6850797335306804
    	round 6: 1.7002071539560955
    	round 7: 1.7214779456456502
    	round 8: 1.7428276141484578
    	round 9: 1.7617273728052776
    	round 10: 1.7799379030863445
    History (metrics, distributed, evaluate):
    {'accuracy': [(1, 0.39839839935302734),
                  (2, 0.3903903861840566),
                  (3, 0.38838838537534076),
                  (4, 0.38538538416226703),
                  (5, 0.3823823829491933),
                  (6, 0.37837838133176166),
                  (7, 0.36936936775843304),
                  (8, 0.36136136452356976),
                  (9, 0.3553553521633148),
                  (10, 0.353353351354599)]}
    ```      
- steps_per_epoch=3
    ```
    Run finished 10 round(s) in 53.10s
    History (loss, distributed):
    	round 1: 1.6459236145019531
    	round 2: 1.6486506859461467
    	round 3: 1.635912815729777
    	round 4: 1.6640913089116414
    	round 5: 1.6798454920450847
    	round 6: 1.6940824588139851
    	round 7: 1.7212083339691162
    	round 8: 1.7342634598414104
    	round 9: 1.7539432446161907
    	round 10: 1.7682697772979736
    History (metrics, distributed, evaluate):
    {'accuracy': [(1, 0.3933933973312378),
                  (2, 0.3943943878014882),
                  (3, 0.38938939571380615),
                  (4, 0.3923923969268799),
                  (5, 0.3853853940963745),
                  (6, 0.37837838133176166),
                  (7, 0.36636637647946674),
                  (8, 0.36936936775843304),
                  (9, 0.353353351354599),
                  (10, 0.3553553521633148)]}
    ```
- steps_per_epoch=5
    ```
    Run finished 10 round(s) in 54.88s
    History (loss, distributed):
    	round 1: 1.6016506751378377
    	round 2: 1.6270814339319866
    	round 3: 1.6338967482248943
    	round 4: 1.6256579160690308
    	round 5: 1.6461429595947266
    	round 6: 1.6543954213460286
    	round 7: 1.6736584107081096
    	round 8: 1.661761959393819
    	round 9: 1.6853443384170532
    	round 10: 1.7279966473579407
    History (metrics, distributed, evaluate):
    {'accuracy': [(1, 0.41841842730840045),
                  (2, 0.4014014005661011),
                  (3, 0.40840840339660645),
                  (4, 0.39939939975738525),
                  (5, 0.41041040420532227),
                  (6, 0.39839839935302734),
                  (7, 0.39039039611816406),
                  (8, 0.39039039611816406),
                  (9, 0.3933933973312378),
                  (10, 0.38588589429855347)]}
    ```

In [16]:
num_rounds = 10
NUM_CLIENTS = 3
steps_per_epoch = 5

class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data  # tuple (x_train, y_train)
        self.val_data = val_data      # tuple (x_val, y_val)

    def get_parameters(self, config):
        # Return the current local model parameters
        return self.model.get_weights()

    def fit(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=1,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        # Return updated parameters and number of training examples
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        # Return loss, number of evaluation examples, and metrics
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

def client_fn(context: Context) -> fl.client.Client:
    """Create a Flower client for a given simulated node."""
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)


    # Each client gets its own model instance
    model = generate_ann()

    # Select this client's local data partition
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    # Return a Flower client
    return MyClient(model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    initial_weights = model.get_weights()
    del model  # free memory early

    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(initial_weights)

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=num_rounds)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated nodes/clients
)

INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
(pid=169470) 2026-03-20 16:16:51.733420: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
(pid=169470) 2026-03-20 16:16:51.798133: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(pid=169470) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
(ClientAppActor pid=169450) 2026-03-20 16:16:56.633035: E external/local_xla/xla/stream_executor/cuda/cuda_platf

# Updating Parameters

The core mechanism behind federated learning is the exchange of model information between the server and the clients. At the beginning of each communication round, the server sends the current **global** model parameters to the selected clients. Each client loads these parameters into its local model, performs local training on its private data (thereby updating the parameters), and sends an update back to the server. The server then aggregates the updates to produce the next version of the global model.

Depending on the algorithm and system design, clients may send back full model parameters, parameter deltas, or gradients. In many baseline approaches (including FedAvg), exchanging model parameters (or equivalent updates) is the most common choice.

This “set parameters → train locally → get updated parameters” abstraction is compatible with a wide range of ML frameworks. It fits naturally with functional-style workflows in **PyTorch** or **JAX**, and it also integrates well with stateful frameworks such as **TensorFlow/Keras** as well as libraries like **scikit-learn**. For doing that you would require re implementing how you pull the model for example with some code similar to:

In [17]:
from flwr.common import Metrics
from typing import List, Tuple, Dict
import numpy as np

def get_parameters(model) -> List[np.ndarray]:
    """Extract model parameters as a list of NumPy arrays."""
    return model.get_weights()

def set_parameters(model, parameters: List[np.ndarray]):
    """Load parameters into a Keras model."""
    model.set_weights(parameters)
    return model

def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    """Weighted average of client accuracies (by number of examples)."""
    if not metrics:
        return {}
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}


### Exercise

Now is your turn, why not you try to run your own architecture and reimplement the precious function with the about ones (get_parameters, set_parameter, weigthed_average). Beaware of the high requirements when we are in a simulated environment.

**Note**: The objective of this exercise is to bring together the various topics covered by this notebook. **This cell should contain all the necessary code for autonomous execution using Flower simulation**.

In [18]:
import tensorflow as tf

# Code to load the dataset
import numpy as np

NUM_CLIENTS = 5
NUM_ROUNDS = 10

def unison_shuffled_copies(a: np.ndarray, b: np.ndarray, seed: int = RANDOM_SEED):
    assert len(a) == len(b)
    rng = np.random.default_rng(seed)
    p = rng.permutation(len(a))
    return a[p], b[p]

def split_index(a, n):
    s = np.array_split(np.arange(len(a)), n)
    return s


def generate_ann():
     #TODO
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(32, activation="softmax")
    ])
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )
    return model

# Code to load the dataset
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)


#TODO Client, client_fn, Server and simulation

class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data  # tuple (x_train, y_train)
        self.val_data = val_data      # tuple (x_val, y_val)

    def get_parameters(self, config):
        # Return the current local model parameters
        return self.model.get_weights()

    def fit(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=1,
            batch_size=32,
            steps_per_epoch=5,
            verbose=0,
        )

        # Return updated parameters and number of training examples
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        # Update local model with global parameters
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        # Return loss, number of evaluation examples, and metrics
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

def client_fn(context: Context) -> fl.client.Client:
    """Create a Flower client for a given simulated node."""
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)


    # Each client gets its own model instance
    model = generate_ann()

    # Select this client's local data partition
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    # Return a Flower client
    return MyClient(model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)

def server_fn(context: Context):
    # Instantiate the model to obtain initial parameters
    model = generate_ann()
    initial_weights = model.get_weights()
    del model  # free memory early

    # Convert to flwr Parameters
    initial_parameters = fl.common.ndarrays_to_parameters(initial_weights)

    # Create FedAvg strategy
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=max(1, NUM_CLIENTS // 2),
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
        evaluate_metrics_aggregation_fn=weighted_average,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,  # number of simulated nodes/clients
)

/home/arebla/OneDrive/MSc/ML2/venv/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")
INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
(pid=175897) 2026-03-20 16:17:36.058834: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
(pid=175897) 2026-03-20 16:17:36.145953: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized 

# Aggregation

To conclude this lesson, let's take a closer look at the key point of these strategies, which is the aggregation algorithm. These algorithms are responsible for combining the updates from the clients to generate the global model, and they are defined in the strategies as we have seen. Generally speaking, there are several types of aggregation that can be used in federated learning (Reddi et. al, 2020).  

Here are the different types of aggregation that can be used in federated learning:

* Federated averaging (`flwr.server.strategy.FedAvg`): In this approach, each device computes an update to the model parameters based on its local data, and these updates are then averaged together to create the global model. This approach is simple and effective, but it can be sensitive to the size of the updates and the quality of the data on each device.

* Federated weighted averaging: This approach is similar to federated averaging, but each device's update is given a different weight based on the size of its data set or the quality of its data. This can help to give more influence to devices with larger or higher-quality data.

* Federated averaging with momentum (`flwr.server.strategy.FedAvgM`): This approach is similar to federated averaging, but it incorporates a momentum term in order to smooth out the updates and help the model converge more quickly.

* Federated stochastic gradient descent(`flwr.server.strategy.FedAdagrad`): In this approach, each device computes an update to the model parameters based on a small batch of its local data, rather than the entire data set. This can help to reduce the communication overhead and improve the convergence rate of the model.

* Federated ADAM (`flwr.server.strategy.FedAdam`): This approach is a variant of federated stochastic gradient descent that uses the ADAM optimization algorithm to adaptively adjust the learning rate based on the gradient and second moment estimates.



All of the previously mentioned aggregation methods, except for Federated Weighted Averaging, are implemented in the `flwr` framework and can be used with the different strategies. Additionally, there are other less common aggregation methods that can be employed. The choice of aggregation method will ultimately depend on the specific characteristics of the data and the requirements of the task at hand.


## Running in terminal
The notebook runs the Flower App directly using `run_simulation(...)`. In practice, Flower Apps are commonly executed from the terminal using the Flower CLI (`flwr run`). This is also the recommended workflow in recent Flower versions.

A minimal project structure could look like this:

my_flower_app
>pyproject.toml

>server_app.py
>
>client_app.py

### 1) `client_app.py` and `server_app.py`

These files should define the `client_app` and `server_app` (the same objects we created in the notebook), for example:

- `client_app = ClientApp(client_fn=...)`
- `server_app = ServerApp(server_fn=...)`

### 2) `pyproject.toml`

Flower Apps use `pyproject.toml` to define dependencies and how to run the app. The easiest way to generate a valid template is:

```bash
flwr new
```

To create the template, you will need to select the flower App (e.g., @flwrlabs/quickstart-tensorflow). 
This will generate a subfolder named after the selected Flower App.

After that, you must install its dependencies 
```bash
cd quickstart-tensorflow && pip install -e .
```

Simply to execute the code you have to put

```bash
flwr run .
```

You can run a basic example directly from your terminal by executing the defined commands and selecting @flwrlabs/quickstart-tensorflow. The generated files work out of the box, so no additional code is necessary. Once it's running, feel free to inspect the file structure created by Flower.




## Quick check (self-assessment)

Answer briefly:

1. **What are two reasons why data cannot be centralized?**  
   Data cannot always be centralized because of privacy/legal restrictions,the data may belong to different organizations or institutions not allowed to share it with each other or thids parties, or the costs and impracticality of uploading massive datasets to a central server.

2. **In FL, what does the server aggregate?**  
   In federated learning, the server aggregates the model updates (weights) or parameters received from the clients.

3. **Why can FL converge more slowly than centralized learning?**  
   FL can converge more slowly because the model is trained in separate clients with different local data, and the server only combines updates after each communication round, so progress per round is noisier than a full-data gradient step.

4. **What does Flower provide, and what do you still need to implement yourself?**  
    Flower provides the infrastructure: client/server coordination, communication rounds, parameter serialization, aggregation strategies... But the ML logic needs to be implemented (model architecture, loss function, optimizer...)



### References and further reading

#### Foundational papers
- McMahan, B., Moore, E., Ramage, D., Hampson, S., & y Arcas, B. A. (2017).  
  **Communication-Efficient Learning of Deep Networks from Decentralized Data**.  
  Proceedings of AISTATS.  
  *(Introduces FedAvg and the modern Federated Learning paradigm)*

- Kairouz, P. et al. (2021).  
  **Advances and Open Problems in Federated Learning**.  
  Foundations and Trends® in Machine Learning.  
  *(Comprehensive overview of challenges, theory, and open research problems)*

---

#### Optimization and learning under heterogeneity
- Li, Y., Bonawitz, K., & Talwar, K. (2020).  
  **FedProx: An Optimizer for Communication-Efficient Federated Learning**.  
  arXiv preprint arXiv:2002.04283.  
  *(Addresses client heterogeneity by adding a proximal term to FedAvg)*

- Reddi, S., Charles, Z., Zaheer, M., Garrett, Z., Rush, K., Konečný, J., Kumar, S., & McMahan, H. B. (2020).  
  **Adaptive Federated Optimization**.  
  arXiv preprint arXiv:2003.00295.  
  *(Extends adaptive optimizers such as Adam and Yogi to the federated setting)*

- Yoon, J., Hard, A., Konečný, J., McMahan, H. B., & Sohl-Dickstein, J. (2018).  
  **Federated Regression: A Simple and Scalable Method for Heterogeneous Federated Learning**.  
  arXiv preprint arXiv:1812.03862.  
  *(Early work addressing statistical heterogeneity in federated settings)*

---

#### Practical frameworks
- Flower Documentation  
  https://flower.ai/docs/  
  *(Official documentation of the Flower framework used in this course)*

- Flower GitHub Repository  
  https://github.com/adap/flower  
  *(Reference implementations and examples)*

---

#### Surveys and applied perspectives
- Li, T., Sahu, A. K., Talwalkar, A., & Smith, V. (2020).  
  **Federated Learning: Challenges, Methods, and Future Directions**.  
  IEEE Signal Processing Magazine.  
  *(Survey focused on optimization and system challenges in FL)*

- Bonawitz, K. et al. (2019).  
  **Towards Federated Learning at Scale: System Design**.  
  Proceedings of MLSys.  
  *(System-level view of large-scale FL deployments)*

---

#### Optional (advanced topics)
- Truex, S. et al. (2019).  
  **A Hybrid Approach to Privacy-Preserving Federated Learning**.  
  *(Privacy risks and mitigation strategies)*

- Hard, A., Konečný, J., McMahan, H. B., Richemond-Barakat, C., Sivek, J. S., & Talwar, K. (2018).  
  **Federated Learning: Strategies for Improving Communication Efficiency.**  
  arXiv preprint arXiv:1812.02903.  
  *(Introduces fundamental communication-efficiency strategies in FL)*

---

*You are not expected to read all these references in detail.  
They are provided for context and for students interested in deeper theoretical or applied aspects of Federated Learning.*
